# Part 2: Fine-tuning Stable Diffusion 1.5 on Naruto BLIP Captions

DATA 266 Lab 2, Part 2: one notebook for the whole part. Run it with **Runtime → Restart session and run all**.

| Section | What it does |
|---|---|
| 1. Baseline | 2.1 data preparation, 20 fixed prompts, base model (SD 1.5, no fine-tuning): 40 images, KID, CLIP, LPIPS |
| 2. Run 1 | LoRA rank 4: training, 40 images, metrics vs base |
| 3. Run 2 | LoRA rank 32, same data and step budget: training, 40 images, metrics vs base |
| 4. Final | choice of final run, grids, blinded human evaluation, **results table baseline vs final**, failure modes, safety, model card |

| Measurement | Part 2 definition |
|---|---|
| Primary | KID vs test images; CLIP score |
| Secondary | Diversity: LPIPS between seeds |
| Human | Blinded 25 pairs, 3+ raters; win rate |
| Efficiency | GPU-hours per run, peak memory |
| Baseline | Base model |

Everything is saved to Google Drive (`MyDrive/data266-lab2/`). With `REUSE = True`, finished steps (cleaned data, trained adapters, generated images, human-eval pairs) are loaded from Drive instead of being redone; the metrics are recomputed every run.

**Base model.** `stable-diffusion-v1-5/stable-diffusion-v1-5` is a *latent* diffusion model (Rombach et al., 2022): a VAE compresses each 512×512 image to a 64×64×4 latent, a CLIP text encoder reads the prompt, and a UNet removes noise step by step, guided by the text.

**LoRA** (Hu et al., 2021) freezes the original weight *W* and learns a small update *ΔW = B·A* of rank *r*. Run 1 uses r = 4 and Run 2 uses r = 32; everything else is identical.

**Memorization** (Carlini et al., 2023) is checked in section 4 with CLIP nearest neighbours.

In [ ]:
%pip install -q diffusers==0.31.0 transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 huggingface_hub==0.26.2 torchmetrics==1.6.0 torch-fidelity==0.3.0 lpips==0.1.4 imagehash==4.3.1

In [ ]:
import subprocess, platform, random, os
import numpy as np, torch, diffusers, transformers, peft, torchmetrics, datasets

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "no git repo here"

print("commit      ", commit)
print("python      ", platform.python_version())
for m in [torch, diffusers, transformers, peft, torchmetrics, datasets]:
    print(f"{m.__name__:12s}", m.__version__)
print("gpu         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("seed        ", SEED)

In [ ]:
import json, time, gc, hashlib, io, textwrap
import pandas as pd
import matplotlib.pyplot as plt
import imagehash
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
from datasets import load_dataset, Image as HFImage, Dataset as HFDataset
from huggingface_hub import dataset_info, HfApi, login, ModelCard
from transformers import CLIPImageProcessor, CLIPTokenizer, CLIPTextModel, CLIPModel, CLIPProcessor
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler, UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.multimodal.clip_score import CLIPScore
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
import lpips

In [ ]:
%config InlineBackend.figure_format = "jpeg"
plt.rcParams["figure.dpi"] = 72
plt.rcParams["savefig.dpi"] = 72

In [ ]:
device = "cuda"
BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DATASET = "lambdalabs/naruto-blip-captions"
RES = 512
STEPS = 3000
BATCH = 4
LR = 1e-4
REUSE = True
RETRAIN = False
FINAL_OVERRIDE = None
HF_USER = "your-hf-username"
UPLOAD = False

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/data266-lab2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = ROOT / "eval"
WORK = ROOT / "work"
DATA = WORK / "p2_data"
RUNS_DIR = WORK / "p2_runs"
GEN_DIR = WORK / "p2_gen"
HUMAN_DIR = WORK / "p2_human"
LOGS = ROOT / "logs"
for d in [EVAL, DATA, RUNS_DIR, GEN_DIR, HUMAN_DIR, LOGS]:
    d.mkdir(parents=True, exist_ok=True)
print("saving to", ROOT)

In [ ]:
LOG_FILE = LOGS / "p2_run_log.txt"

def log(*args, file=LOG_FILE):
    line = time.strftime("%Y-%m-%d %H:%M:%S") + " | " + " ".join(str(a) for a in args)
    print(line)
    with open(file, "a") as f:
        f.write(line + "\n")

log("=" * 60)
log("part 2 submission notebook | seed", SEED, "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
log("commit", commit, "| torch", torch.__version__, "| diffusers", diffusers.__version__, "| peft", peft.__version__)

# Section 1: Baseline (base model, no fine-tuning)

In [ ]:
CLEAN_FILE = EVAL / "p2_data_cleaning.json"
DATA_CACHED = REUSE and all(f.exists() for f in [CLEAN_FILE, DATA / "train_clean.csv", DATA / "test.csv"])
print("reuse saved data from Drive:", DATA_CACHED)

## 2.1 Data preparation

Steps: load the dataset → remove broken images and duplicates → split 90/10 with seed 266 → crop and resize to 512×512 → NSFW filter on training images → describe the data.

- **Corrupted:** the file can't be opened, is tiny (<64 px) or has an empty caption.
- **Exact duplicate:** same file bytes (MD5 hash).
- **Near duplicate:** perceptual hash differs by ≤ 2 bits, i.e. the same picture saved slightly differently.

In [ ]:
if DATA_CACHED:
    print("skipping download: cleaned data already saved in", DATA)
else:
    raw = load_dataset(DATASET, split="train").cast_column("image", HFImage(decode=False))
    print(raw)

In [ ]:
info = dataset_info(DATASET)
DATA_LICENSE = (info.card_data or {}).get("license")
if DATA_LICENSE is None:
    DATA_LICENSE = "not stated on the dataset card; images from narutopedia.com (Naruto is copyrighted), non-commercial research use only"
print("dataset license:", DATA_LICENSE)

In [ ]:
def img_bytes(x):
    if x.get("bytes"):
        return x["bytes"]
    return open(x["path"], "rb").read()

In [ ]:
if DATA_CACHED:
    cleaning = json.loads(CLEAN_FILE.read_text())
    print("loaded cleaning report from", CLEAN_FILE)
else:
    keep, corrupted, exact_dup, near_dup = [], [], [], []
    seen_md5 = set()
    seen_phash = []

    for i, row in enumerate(raw):
        try:
            b = img_bytes(row["image"])
            Image.open(io.BytesIO(b)).verify()
            im = Image.open(io.BytesIO(b)).convert("RGB")
            if min(im.size) < 64 or not str(row["text"]).strip():
                raise ValueError
        except Exception:
            corrupted.append(i)
            continue

        md5 = hashlib.md5(b).hexdigest()
        if md5 in seen_md5:
            exact_dup.append(i)
            continue

        ph = imagehash.phash(im)
        if any(ph - h <= 2 for h in seen_phash):
            near_dup.append(i)
            continue

        seen_md5.add(md5)
        seen_phash.append(ph)
        keep.append(i)

    cleaning = {
        "dataset": DATASET,
        "license": DATA_LICENSE,
        "original": len(raw),
        "corrupted": len(corrupted),
        "exact_duplicates": len(exact_dup),
        "near_duplicates": len(near_dup),
        "kept": len(keep),
        "removed_ids": {"corrupted": corrupted, "exact_dup": exact_dup, "near_dup": near_dup},
    }

In [ ]:
pd.Series({
    "original": cleaning["original"],
    "corrupted / empty caption": cleaning["corrupted"],
    "exact duplicates": cleaning["exact_duplicates"],
    "near duplicates (phash<=2)": cleaning["near_duplicates"],
    "removed total": cleaning["original"] - cleaning["kept"],
    "kept": cleaning["kept"],
}, name="count")

In [ ]:
if not DATA_CACHED:
    split = raw.select(keep).train_test_split(test_size=0.1, seed=SEED)
    train_raw = split["train"]
    test_raw = split["test"]
    log("cleaning | original", cleaning["original"], "| corrupted", cleaning["corrupted"], "| exact dup",
        cleaning["exact_duplicates"], "| near dup", cleaning["near_duplicates"], "| kept", cleaning["kept"])
    log("split | train", len(train_raw), "| test", len(test_raw))

In [ ]:
def to_square(b, size=RES):
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return im.resize((size, size), Image.BICUBIC)

In [ ]:
def dump(ds, name):
    out = DATA / name
    out.mkdir(exist_ok=True)
    meta = []
    for j, row in enumerate(ds):
        p = out / f"{j:04d}.png"
        if not p.exists():
            to_square(img_bytes(row["image"])).save(p)
        meta.append({"file": str(p), "text": row["text"]})
    df = pd.DataFrame(meta)
    df.to_csv(DATA / f"{name}.csv", index=False)
    return df

In [ ]:
if DATA_CACHED:
    train_df = pd.read_csv(DATA / "train.csv")
    test_df = pd.read_csv(DATA / "test.csv")
else:
    train_df = dump(train_raw, "train")
    test_df = dump(test_raw, "test")
print("image size", Image.open(train_df.file[0]).size, "| train", len(train_df), "| test", len(test_df))

### NSFW filter on training images

We use the official Stable Diffusion safety checker (`CompVis/stable-diffusion-safety-checker`), a CLIP-based classifier. Flagged training images are removed. It is known to be conservative on drawn characters, so some flags are probably false positives.

In [ ]:
safety = None
safety_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
def nsfw_flags(paths, bs=32):
    global safety
    if safety is None:
        safety = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker").to(device)
    flags = []
    for k in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[k:k + bs]]
        clip_in = safety_proc(ims, return_tensors="pt").pixel_values.to(device)
        arr = np.stack([np.asarray(im, dtype=np.float32) / 255 for im in ims])
        _, has = safety(images=arr, clip_input=clip_in)
        flags += [bool(h) for h in has]
    return flags

In [ ]:
if DATA_CACHED:
    n_train_nsfw = cleaning["train_nsfw_removed"]
    print("loaded NSFW result from Drive")
else:
    train_df["nsfw"] = nsfw_flags(train_df.file.tolist())
    n_train_nsfw = int(train_df.nsfw.sum())
print("training images flagged NSFW:", n_train_nsfw)

In [ ]:
if DATA_CACHED:
    train_df = pd.read_csv(DATA / "train_clean.csv")
else:
    train_df = train_df[~train_df.nsfw].reset_index(drop=True)
    train_df.to_csv(DATA / "train_clean.csv", index=False)
    log("nsfw filter on train | flagged", n_train_nsfw, f"({n_train_nsfw / (len(train_df) + n_train_nsfw):.1%})",
        "| train left", len(train_df))
print("train after NSFW filter:", len(train_df))

In [ ]:
cleaning.update({"license": DATA_LICENSE, "train": len(train_df), "test": len(test_df),
                 "train_nsfw_removed": n_train_nsfw, "split_seed": SEED, "resolution": RES})
CLEAN_FILE.write_text(json.dumps(cleaning, indent=2))
{k: v for k, v in cleaning.items() if k != "removed_ids"}

### Dataset description

In [ ]:
words = train_df.text.str.split().str.len()
print("caption length (words): mean %.1f, min %d, max %d" % (words.mean(), words.min(), words.max()))

In [ ]:
train_df.text.str.split().str[:2].str.join(" ").value_counts().head(8)

In [ ]:
{k: round(train_df.text.str.contains(rf"\b{k}\b").mean(), 3) for k in ["man", "woman", "boy", "girl"]}

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes.flat, train_df.sample(8, random_state=SEED).iterrows()):
    ax.imshow(Image.open(r.file))
    ax.axis("off")
    ax.set_title(textwrap.fill(r.text, 30), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
names = ["naruto", "sasuke", "sakura", "kakashi", "itachi", "hinata", "gaara", "jiraiya", "orochimaru", "madara"]
txt = train_df.text.str.lower()
cap_words = txt.str.split().str.len()
name_share = txt.str.contains("|".join(names)).mean()
share = {k: txt.str.contains(rf"\b{k}\b").mean() for k in ["man", "woman", "boy", "girl"]}
multi_share = txt.str.contains(r"\b(?:two|three|group|people|couple)\b").mean()

dataset_txt = (
    f"Dataset: Naruto BLIP Captions ({DATASET}), images of characters from the Naruto anime collected from narutopedia.com, "
    f"each with a caption generated automatically by BLIP. After cleaning: {len(train_df)} train and {len(test_df)} test images, "
    f"center-cropped and resized to {RES}x{RES}. License: {DATA_LICENSE}."
)
limit1 = (
    f"Limitation 1, machine captions: captions are short (mean {cap_words.mean():.1f} words, max {cap_words.max()}) and generic. "
    f"Only {100 * name_share:.1f}% name a character, so the model learns the art style more than identities, "
    f"and captions are BLIP guesses rather than human labels."
)
limit2 = (
    f"Limitation 2, unbalanced content: {100 * share['man']:.1f}% of captions mention a man vs {100 * share['woman']:.1f}% a woman, "
    f"{100 * share['girl']:.1f}% a girl and {100 * share['boy']:.1f}% a boy, and only {100 * multi_share:.1f}% describe more than one person, "
    f"so multi-character and compositional prompts are under-represented."
)
print(dataset_txt, limit1, limit2, sep="\n\n")
(EVAL / "p2_dataset_notes.md").write_text("\n\n".join([dataset_txt, limit1, limit2]))

## Fixed evaluation prompts (written before training, saved to `eval/p2_prompts.json`)

- **10 in-domain:** real captions from the *test* set, so the model has never seen these exact image–caption pairs.
- **5 out-of-domain:** things that are not in Naruto (cars, food, dogs). These test whether fine-tuning broke general ability.
- **5 compositional:** several objects with colors and positions. These test attribute binding, which SD 1.5 is weak at.

If the file already exists, it is reused, so the prompts never change after the first run.

In [ ]:
in_domain = test_df.sample(10, random_state=SEED).text.tolist()

ood = [
    "a red sports car parked on a rainy city street",
    "a lighthouse on a rocky cliff at sunset",
    "a bowl of fruit on a wooden kitchen table",
    "an astronaut floating above the earth",
    "a golden retriever running on a beach",
]

comp = [
    "two characters, one in red and one in blue",
    "a girl with pink hair standing to the left of a boy with yellow hair",
    "three ninjas standing in a row, the middle one wearing a mask",
    "a man holding a sword in his left hand and a scroll in his right hand",
    "a small orange fox sitting on top of a large green frog",
]

In [ ]:
prompt_file = EVAL / "p2_prompts.json"
if prompt_file.exists():
    prompts = json.loads(prompt_file.read_text())
    print("using saved prompts from", prompt_file)
else:
    items = [("in_domain", p) for p in in_domain] + [("ood", p) for p in ood] + [("compositional", p) for p in comp]
    prompts = [{"id": i, "category": c, "prompt": p} for i, (c, p) in enumerate(items)]
    prompt_file.write_text(json.dumps(prompts, indent=2))
    log("prompts saved before training |", len(prompts), "prompts | md5", hashlib.md5(prompt_file.read_bytes()).hexdigest())
pd.DataFrame(prompts)

In [ ]:
counts = pd.Series([p["category"] for p in prompts]).value_counts()
assert len(prompts) == 20
assert counts["in_domain"] == 10 and counts["ood"] == 5 and counts["compositional"] == 5
counts

## Baseline generation (base model, no fine-tuning)

Sampler settings are fixed here and are reused unchanged by both runs: DPM-Solver++, 30 steps, CFG 7.5, 512×512, seeds 0 and 1. That gives 20 prompts × 2 seeds = 40 images.

In [ ]:
GEN = dict(num_inference_steps=30, guidance_scale=7.5, height=RES, width=RES)
(EVAL / "p2_sampler.json").write_text(json.dumps({"scheduler": "DPMSolverMultistepScheduler", **GEN}, indent=2))
print("DPMSolverMultistepScheduler", GEN)

In [ ]:
META_FILE = GEN_DIR / "base_metadata.csv"
GEN_STATS = GEN_DIR / "base_gen_stats.json"
GEN_CACHED = False
if REUSE and META_FILE.exists() and GEN_STATS.exists() and json.loads(GEN_STATS.read_text()).get("version") == 2:
    base = pd.read_csv(META_FILE)
    GEN_CACHED = len(base) == 40 and all(Path(f).exists() for f in base.file)
print("reuse saved base images from Drive:", GEN_CACHED)

In [ ]:
if not GEN_CACHED:
    pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None,
                                                   requires_safety_checker=False).to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.set_progress_bar_config(disable=True)

In [ ]:
def generate(model):
    rows = []
    for p in prompts:
        for seed in (0, 1):
            path = GEN_DIR / f"{model}_p{p['id']:02d}_s{seed}.png"
            t0 = time.time()
            img = pipe(p["prompt"], generator=torch.Generator(device).manual_seed(seed), **GEN).images[0]
            sec = time.time() - t0
            img.save(path)
            rows.append({"model": model, "prompt_id": p["id"], "category": p["category"],
                         "prompt": p["prompt"], "seed": seed, "file": str(path), "sec": sec})
    return rows

In [ ]:
if GEN_CACHED:
    gen_peak = json.loads(GEN_STATS.read_text())["peak_gb"]
    print("loaded", len(base), "base images from", GEN_DIR)
else:
    safety = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = pd.DataFrame(generate("base"))
    gen_peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    base.to_csv(META_FILE, index=False)
    GEN_STATS.write_text(json.dumps({"peak_gb": gen_peak, "version": 2}))
    log("baseline generation |", len(base), "images | median sec/img", round(base.sec.median(), 2))
print("images:", len(base), "| median sec/img %.2f | peak GPU mem %.2f GB" % (base.sec.median(), gen_peak))

In [ ]:
def show_row(ids, seed=0, title=None):
    fig, axes = plt.subplots(1, len(ids), figsize=(3.2 * len(ids), 3.8))
    for ax, pid in zip(axes, ids):
        f = base[(base.prompt_id == pid) & (base.seed == seed)].file.iloc[0]
        ax.imshow(Image.open(f))
        ax.axis("off")
        ax.set_title(textwrap.fill(f"[{pid}] {prompts[pid]['prompt']}", 26), fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "in_domain"][:5], title="base model, in-domain (seed 0)")

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "ood"], title="base model, out-of-domain (seed 0)")

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "compositional"], title="base model, compositional (seed 0)")

## Baseline metrics

| Metric | What it measures | Better |
|---|---|---|
| KID | distance between Inception features of the 20 in-domain generations and the test images | lower |
| CLIP score | how well each image matches its prompt (100 × cosine similarity) | higher |
| LPIPS | perceptual distance between seed 0 and seed 1 of the same prompt (diversity) | higher = more varied |

KID is used instead of FID because it is unbiased with small samples. With only 20 images it is still noisy, so we also report the standard deviation over 100 subsets.

In [ ]:
def load_uint8(files, size=None):
    ims = [Image.open(f).convert("RGB") for f in files]
    if size:
        ims = [im.resize((size, size), Image.BICUBIC) for im in ims]
    return torch.from_numpy(np.stack([np.asarray(im) for im in ims])).permute(0, 3, 1, 2)

In [ ]:
real = load_uint8(test_df.file, 299)
fake = load_uint8(base[base.category == "in_domain"].file, 299)
real.shape, fake.shape

In [ ]:
kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)), subsets=100).to(device)
torch.manual_seed(SEED)
for k in range(0, len(real), 32):
    kid.update(real[k:k + 32].to(device), real=True)
kid.update(fake.to(device), real=False)
kid_mu, kid_sd = [v.item() for v in kid.compute()]
log(f"baseline KID = {kid_mu:.4f} ± {kid_sd:.4f} (fake={len(fake)}, real={len(real)})")

In [ ]:
clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to(device)

scores = []
for r in base.itertuples():
    scores.append(clip_metric(load_uint8([r.file])[0].to(device), r.prompt).item())
    clip_metric.reset()
base["clip"] = scores
base.groupby("category").clip.mean().round(2)

In [ ]:
lp = lpips.LPIPS(net="alex").to(device)

def to_lp(f):
    x = np.asarray(Image.open(f).convert("RGB"), dtype=np.float32) / 127.5 - 1
    return torch.from_numpy(x).permute(2, 0, 1)[None].to(device)

In [ ]:
div = []
for pid, g in base.groupby("prompt_id"):
    a, b = g.sort_values("seed").file.tolist()
    with torch.no_grad():
        d = lp(to_lp(a), to_lp(b)).item()
    div.append({"model": "base", "prompt_id": pid, "category": g.category.iloc[0], "lpips": d})
div = pd.DataFrame(div)
div.groupby("category").lpips.mean().round(3)

In [ ]:
if "nsfw" not in base.columns:
    base["nsfw"] = nsfw_flags(base.file.tolist())
print("base generations flagged NSFW:", int(base.nsfw.sum()), "/", len(base))
base[base.nsfw][["prompt_id", "seed", "category", "prompt"]]

## Baseline results table

In [ ]:
baseline = pd.Series({
    "KID in-domain (lower better)": kid_mu,
    "KID std": kid_sd,
    "CLIP all": base["clip"].mean(),
    "CLIP in_domain": base[base.category == "in_domain"]["clip"].mean(),
    "CLIP ood": base[base.category == "ood"]["clip"].mean(),
    "CLIP compositional": base[base.category == "compositional"]["clip"].mean(),
    "LPIPS diversity": div.lpips.mean(),
    "NSFW flagged": int(base.nsfw.sum()),
    "median sec/img (gen)": base.sec.median(),
    "GPU-hours (train)": 0.0,
    "peak GPU mem GB (train)": float("nan"),
    "GPU-hours (generate 40 imgs)": base.sec.sum() / 3600,
    "peak GPU mem GB (generate)": gen_peak,
}, name="base").round(4)
baseline

In [ ]:
baseline.to_frame().to_csv(EVAL / "p2_baseline_results.csv")
base.to_csv(GEN_DIR / "base_metadata.csv", index=False)
div.to_csv(GEN_DIR / "base_lpips.csv", index=False)
log("baseline results\n" + baseline.to_string())

## Files saved by section 1

In [ ]:
for f in [DATA / "train_clean.csv", DATA / "test.csv", EVAL / "p2_prompts.json", EVAL / "p2_sampler.json",
          EVAL / "p2_data_cleaning.json", EVAL / "p2_baseline_results.csv", GEN_DIR / "base_metadata.csv",
          GEN_DIR / "base_lpips.csv"]:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")

### Free GPU memory before the next section

In [ ]:
pipe = safety = kid = clip_metric = lp = None
gc.collect()
torch.cuda.empty_cache()
print("GPU memory still in use: %.2f GB" % (torch.cuda.memory_allocated() / 1024 ** 3))

# Section 2: Run 1, LoRA rank 4

LoRA rank 4, alpha 4, on the UNet attention layers (q, k, v, out). Same data, steps, batch and learning rate as Run 2.

In [ ]:
RUN = "run1"
RANK = 4
RUN_DIR = RUNS_DIR / RUN
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = LOGS / f"{RUN}_train_log.txt"
log(f"{RUN} section | rank {RANK} | steps {STEPS} | batch {BATCH} | lr {LR}")

## Load what section 1 saved

In [ ]:
needed = [DATA / "train_clean.csv", DATA / "test.csv", EVAL / "p2_prompts.json", EVAL / "p2_sampler.json",
          EVAL / "p2_baseline_results.csv", GEN_DIR / "base_metadata.csv", GEN_DIR / "base_lpips.csv"]
for f in needed:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")
assert all(f.exists() for f in needed), "run section 1 first"

In [ ]:
train_df = pd.read_csv(DATA / "train_clean.csv")
test_df = pd.read_csv(DATA / "test.csv")
prompts = json.loads((EVAL / "p2_prompts.json").read_text())
sampler = json.loads((EVAL / "p2_sampler.json").read_text())
baseline = pd.read_csv(EVAL / "p2_baseline_results.csv", index_col=0)["base"]
base = pd.read_csv(GEN_DIR / "base_metadata.csv")
base_div = pd.read_csv(GEN_DIR / "base_lpips.csv")
print("train", len(train_df), "| test", len(test_df), "| prompts", len(prompts), "| sampler", sampler)

## 2.2 Training Run 1

One training step:
1. Encode the image to a 64×64×4 latent with the frozen VAE.
2. Encode the caption with the frozen CLIP text encoder.
3. Add random noise at a random timestep *t*.
4. The UNet (with LoRA) predicts that noise.
5. Loss = MSE between the predicted and the real noise. Only the LoRA weights are updated.

Memory savers: fp16 base weights, fp32 LoRA weights, gradient checkpointing, mixed precision.

`RETRAIN = False` reloads the adapter already saved in Drive and prints its saved training log. Set it to `True` to train again (about 75 min on a T4).

In [ ]:
TRAINED = (RUN_DIR / "stats.json").exists() and (RUN_DIR / "pytorch_lora_weights.safetensors").exists()
DO_TRAIN = RETRAIN or not TRAINED
print("adapter saved in Drive:", TRAINED, "| train now:", DO_TRAIN)

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
if DO_TRAIN:
    text_encoder = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to(device)
    vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to(device)
    noise_sched = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
    text_encoder.requires_grad_(False)
    vae.requires_grad_(False)
    print("frozen: text encoder + VAE")
else:
    print("not training: text encoder and VAE not loaded")

In [ ]:
class NarutoTrain(Dataset):
    def __init__(self, df):
        self.df = df
        self.tf = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        ids = tokenizer(r.text, padding="max_length", max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt").input_ids[0]
        return {"pixel_values": self.tf(Image.open(r.file).convert("RGB")), "input_ids": ids}

In [ ]:
train_ds = NarutoTrain(train_df)
len(train_ds)

In [ ]:
def make_lora_unet(rank):
    torch.manual_seed(SEED)
    unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16).to(device)
    unet.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=rank, init_lora_weights="gaussian",
                                target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
    cast_training_params(unet, dtype=torch.float32)
    unet.enable_gradient_checkpointing()
    return unet

In [ ]:
def train(unet, steps=STEPS, lr=LR, bs=BATCH):
    params = [p for p in unet.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)
    lr_sched = get_scheduler("cosine", opt, num_warmup_steps=100, num_training_steps=steps)
    scaler = torch.amp.GradScaler("cuda")
    loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=2,
                        generator=torch.Generator().manual_seed(SEED))

    log(f"{RUN} start | steps {steps} | batch {bs} | lr {lr} | trainable {sum(p.numel() for p in params):,}", file=TRAIN_LOG)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    losses = []
    step = 0
    unet.train()

    while step < steps:
        for batch in loader:
            with torch.no_grad():
                lat = vae.encode(batch["pixel_values"].to(device, torch.float16)).latent_dist.sample()
                lat = lat * vae.config.scaling_factor
                enc = text_encoder(batch["input_ids"].to(device))[0]

            noise = torch.randn_like(lat)
            t = torch.randint(0, noise_sched.config.num_train_timesteps, (lat.shape[0],), device=device).long()
            noisy = noise_sched.add_noise(lat, noise, t)

            with torch.autocast("cuda", dtype=torch.float16):
                pred = unet(noisy, t, enc).sample
            loss = F.mse_loss(pred.float(), noise.float())

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            lr_sched.step()
            opt.zero_grad(set_to_none=True)

            losses.append(loss.item())
            step += 1
            if step % 250 == 0:
                mem = torch.cuda.max_memory_allocated() / 1024 ** 3
                log(f"{RUN} step {step}/{steps} | loss {np.mean(losses[-250:]):.4f} | lr {lr_sched.get_last_lr()[0]:.2e} | "
                    f"{(time.time() - t0) / 60:.1f} min | peak {mem:.2f} GB", file=TRAIN_LOG)
            if step >= steps:
                break

    torch.cuda.synchronize()
    log(f"{RUN} done | {(time.time() - t0) / 60:.1f} min | peak {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB", file=TRAIN_LOG)
    return {
        "rank": RANK,
        "trainable_params": sum(p.numel() for p in params),
        "lr": lr, "steps": steps, "batch_size": bs,
        "peak_mem_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
        "train_sec": time.time() - t0,
        "gpu": torch.cuda.get_device_name(0),
        "losses": losses,
    }

In [ ]:
def save_run(unet, stats):
    lora_sd = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionPipeline.save_lora_weights(RUN_DIR, unet_lora_layers=lora_sd, safe_serialization=True)
    (RUN_DIR / "stats.json").write_text(json.dumps(stats))

In [ ]:
if not DO_TRAIN:
    stats = json.loads((RUN_DIR / "stats.json").read_text())
    print(f"loaded saved {RUN} from {RUN_DIR}\n")
    print(TRAIN_LOG.read_text() if TRAIN_LOG.exists() else "no saved training log")
else:
    unet = make_lora_unet(RANK)
    stats = train(unet)
    save_run(unet, stats)
    del unet
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
sorted(p.name for p in RUN_DIR.iterdir())

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(stats["losses"], alpha=0.25, label="loss per step")
plt.plot(pd.Series(stats["losses"]).rolling(100).mean(), label="rolling mean (100)")
plt.xlabel("step")
plt.ylabel("MSE loss")
plt.title(f"{RUN}: LoRA r={RANK} training loss")
plt.legend()
plt.show()

### Training table (Run 1)

In [ ]:
train_row = pd.Series({
    "method": f"LoRA r={stats['rank']}",
    "trainable_params": f"{stats['trainable_params']:,}",
    "learning_rate": stats["lr"],
    "steps": stats["steps"],
    "batch_size": stats["batch_size"],
    "peak_gpu_mem_GB": round(stats["peak_mem_gb"], 2),
    "train_time_min": round(stats["train_sec"] / 60, 1),
    "gpu_hours": round(stats["train_sec"] / 3600, 3),
    "gpu": stats["gpu"],
}, name=RUN)
train_row.to_frame().to_csv(EVAL / f"p2_{RUN}_training.csv")
log(f"{RUN} | " + " | ".join(f"{k} {v}" for k, v in train_row.items()))
train_row.to_frame()

In [ ]:
repo = f"{HF_USER}/naruto-sd15-lora-r{RANK}"
if UPLOAD:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    api = HfApi()
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=RUN_DIR, allow_patterns=["*.safetensors", "stats.json"])
    log("uploaded", RUN, "->", f"https://huggingface.co/{repo}")
else:
    print("UPLOAD is False: adapter saved only in", RUN_DIR)

## 2.3 Generation with Run 1

Same prompts, seeds (0 and 1) and sampler settings as the baseline, so the only difference from the base images is the LoRA adapter.

In [ ]:
if DO_TRAIN:
    del text_encoder, vae
gc.collect()
torch.cuda.empty_cache()

In [ ]:
GEN = {k: v for k, v in sampler.items() if k != "scheduler"}
META_FILE = GEN_DIR / f"{RUN}_metadata.csv"
GEN_STATS = GEN_DIR / f"{RUN}_gen_stats.json"
GEN_CACHED = False
if REUSE and not DO_TRAIN and META_FILE.exists() and GEN_STATS.exists():
    gen = pd.read_csv(META_FILE)
    GEN_CACHED = len(gen) == 40 and all(Path(f).exists() for f in gen.file)
print("reuse saved", RUN, "images from Drive:", GEN_CACHED, "| sampler", GEN)

In [ ]:
if not GEN_CACHED:
    pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None,
                                                   requires_safety_checker=False).to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.set_progress_bar_config(disable=True)
    pipe.load_lora_weights(str(RUN_DIR))

In [ ]:
if GEN_CACHED:
    gen_peak = json.loads(GEN_STATS.read_text())["peak_gb"]
    print("loaded", len(gen), RUN, "images from", GEN_DIR)
else:
    torch.cuda.reset_peak_memory_stats()
    rows = []
    for p in prompts:
        for seed in (0, 1):
            path = GEN_DIR / f"{RUN}_p{p['id']:02d}_s{seed}.png"
            t0 = time.time()
            img = pipe(p["prompt"], generator=torch.Generator(device).manual_seed(seed), **GEN).images[0]
            sec = time.time() - t0
            img.save(path)
            rows.append({"model": RUN, "prompt_id": p["id"], "category": p["category"],
                         "prompt": p["prompt"], "seed": seed, "file": str(path), "sec": sec})
    gen = pd.DataFrame(rows)
    gen_peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    gen.to_csv(META_FILE, index=False)
    GEN_STATS.write_text(json.dumps({"peak_gb": gen_peak, "version": 2}))
    log(f"{RUN} generation |", len(gen), "images | median sec/img", round(gen.sec.median(), 2))
print("images:", len(gen), "| median sec/img %.2f | peak GPU mem %.2f GB" % (gen.sec.median(), gen_peak))

In [ ]:
def show_pairs(ids, seed=0, title=None):
    fig, axes = plt.subplots(len(ids), 2, figsize=(7, 3.4 * len(ids)))
    for r, pid in enumerate(ids):
        for c, df in enumerate([base, gen]):
            f = df[(df.prompt_id == pid) & (df.seed == seed)].file.iloc[0]
            axes[r, c].imshow(Image.open(f))
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(["base", RUN][c], fontsize=12)
        axes[r, 0].set_ylabel(textwrap.fill(f"[{pid}] {prompts[pid]['prompt']}", 26), fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
grid_ids = [0, 1, 2, 3, 10, 12, 15, 16]
show_pairs(grid_ids, title=f"base vs {RUN} (seed 0)")

In [ ]:
show_pairs(grid_ids[:4], seed=1, title=f"base vs {RUN} (seed 1)")

## 2.4 Metrics for Run 1

In [ ]:
def load_uint8(files, size=None):
    ims = [Image.open(f).convert("RGB") for f in files]
    if size:
        ims = [im.resize((size, size), Image.BICUBIC) for im in ims]
    return torch.from_numpy(np.stack([np.asarray(im) for im in ims])).permute(0, 3, 1, 2)

In [ ]:
real = load_uint8(test_df.file, 299)
fake = load_uint8(gen[gen.category == "in_domain"].file, 299)
real.shape, fake.shape

In [ ]:
kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)), subsets=100).to(device)
torch.manual_seed(SEED)
for k in range(0, len(real), 32):
    kid.update(real[k:k + 32].to(device), real=True)
kid.update(fake.to(device), real=False)
kid_mu, kid_sd = [v.item() for v in kid.compute()]
log(f"{RUN} KID = {kid_mu:.4f} ± {kid_sd:.4f} (base was {baseline['KID in-domain (lower better)']:.4f})")

In [ ]:
clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to(device)

scores = []
for r in gen.itertuples():
    scores.append(clip_metric(load_uint8([r.file])[0].to(device), r.prompt).item())
    clip_metric.reset()
gen["clip"] = scores

In [ ]:
clip_cmp = pd.DataFrame({"base": base.groupby("category").clip.mean(), RUN: gen.groupby("category").clip.mean()})
clip_cmp[f"{RUN} - base"] = clip_cmp[RUN] - clip_cmp["base"]
clip_cmp.round(2)

In [ ]:
lp = lpips.LPIPS(net="alex").to(device)

def to_lp(f):
    x = np.asarray(Image.open(f).convert("RGB"), dtype=np.float32) / 127.5 - 1
    return torch.from_numpy(x).permute(2, 0, 1)[None].to(device)

In [ ]:
div = []
for pid, g in gen.groupby("prompt_id"):
    a, b = g.sort_values("seed").file.tolist()
    with torch.no_grad():
        d = lp(to_lp(a), to_lp(b)).item()
    div.append({"model": RUN, "prompt_id": pid, "category": g.category.iloc[0], "lpips": d})
div = pd.DataFrame(div)
pd.DataFrame({"base": base_div.groupby("category").lpips.mean(), RUN: div.groupby("category").lpips.mean()}).round(3)

In [ ]:
safety = None
safety_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

def nsfw_flags(paths, bs=32):
    global safety
    if safety is None:
        safety = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker").to(device)
    flags = []
    for k in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[k:k + bs]]
        clip_in = safety_proc(ims, return_tensors="pt").pixel_values.to(device)
        arr = np.stack([np.asarray(im, dtype=np.float32) / 255 for im in ims])
        _, has = safety(images=arr, clip_input=clip_in)
        flags += [bool(h) for h in has]
    return flags

if "nsfw" not in gen.columns:
    gen["nsfw"] = nsfw_flags(gen.file.tolist())
print(f"{RUN} generations flagged NSFW:", int(gen.nsfw.sum()), "/", len(gen))
gen[gen.nsfw][["prompt_id", "seed", "category", "prompt"]]

## Results: base vs Run 1

In [ ]:
run_res = pd.Series({
    "KID in-domain (lower better)": kid_mu,
    "KID std": kid_sd,
    "CLIP all": gen["clip"].mean(),
    "CLIP in_domain": gen[gen.category == "in_domain"]["clip"].mean(),
    "CLIP ood": gen[gen.category == "ood"]["clip"].mean(),
    "CLIP compositional": gen[gen.category == "compositional"]["clip"].mean(),
    "LPIPS diversity": div.lpips.mean(),
    "NSFW flagged": int(gen.nsfw.sum()),
    "median sec/img (gen)": gen.sec.median(),
    "GPU-hours (train)": stats["train_sec"] / 3600,
    "peak GPU mem GB (train)": stats["peak_mem_gb"],
    "GPU-hours (generate 40 imgs)": gen.sec.sum() / 3600,
    "peak GPU mem GB (generate)": gen_peak,
}, name=RUN)
results = pd.concat([baseline, run_res], axis=1).round(4)
results

In [ ]:
run_res.round(4).to_frame().to_csv(EVAL / f"p2_{RUN}_results.csv")
gen.to_csv(GEN_DIR / f"{RUN}_metadata.csv", index=False)
div.to_csv(GEN_DIR / f"{RUN}_lpips.csv", index=False)
log(f"{RUN} results\n" + results.to_string())

## Run 1 failure examples

In [ ]:
worst = gen.nsmallest(4, "clip")
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, row in zip(axes, worst.itertuples()):
    ax.imshow(Image.open(row.file))
    ax.axis("off")
    ax.set_title(textwrap.fill(f"CLIP {row.clip:.1f} | {row.prompt}", 34), fontsize=8)
fig.suptitle(f"{RUN}: lowest prompt match")
plt.tight_layout()
plt.show()

In [ ]:
show_pairs([p["id"] for p in prompts if p["category"] == "compositional"][:3], title=f"{RUN}: compositional prompts")

In [ ]:
low = div.nsmallest(2, "lpips")
fig, axes = plt.subplots(len(low), 2, figsize=(7, 3.4 * len(low)))
for r, row in enumerate(low.itertuples()):
    for s in (0, 1):
        f = gen[(gen.prompt_id == row.prompt_id) & (gen.seed == s)].file.iloc[0]
        axes[r, s].imshow(Image.open(f))
        axes[r, s].axis("off")
        axes[r, s].set_title(f"p{row.prompt_id} seed {s}  lpips={row.lpips:.3f}", fontsize=8)
fig.suptitle(f"{RUN}: least diverse prompts")
plt.tight_layout()
plt.show()

**Run 1 observations** (write in your own words after looking at the images):
- ...
- ...

In [ ]:
for f in [RUN_DIR / "pytorch_lora_weights.safetensors", RUN_DIR / "stats.json", EVAL / f"p2_{RUN}_training.csv",
          EVAL / f"p2_{RUN}_results.csv", GEN_DIR / f"{RUN}_metadata.csv", GEN_DIR / f"{RUN}_lpips.csv"]:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")

### Free GPU memory before the next section

In [ ]:
pipe = safety = kid = clip_metric = lp = None
gc.collect()
torch.cuda.empty_cache()
print("GPU memory still in use: %.2f GB" % (torch.cuda.memory_allocated() / 1024 ** 3))

# Section 3: Run 2, LoRA rank 32

LoRA rank 32, alpha 32, on the same UNet attention layers. Same data, steps, batch and learning rate as Run 1, so any difference comes from the rank.

In [ ]:
RUN = "run2"
RANK = 32
RUN_DIR = RUNS_DIR / RUN
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = LOGS / f"{RUN}_train_log.txt"
log(f"{RUN} section | rank {RANK} | steps {STEPS} | batch {BATCH} | lr {LR}")

## Load what section 1 saved

In [ ]:
needed = [DATA / "train_clean.csv", DATA / "test.csv", EVAL / "p2_prompts.json", EVAL / "p2_sampler.json",
          EVAL / "p2_baseline_results.csv", GEN_DIR / "base_metadata.csv", GEN_DIR / "base_lpips.csv"]
for f in needed:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")
assert all(f.exists() for f in needed), "run section 1 first"

In [ ]:
train_df = pd.read_csv(DATA / "train_clean.csv")
test_df = pd.read_csv(DATA / "test.csv")
prompts = json.loads((EVAL / "p2_prompts.json").read_text())
sampler = json.loads((EVAL / "p2_sampler.json").read_text())
baseline = pd.read_csv(EVAL / "p2_baseline_results.csv", index_col=0)["base"]
base = pd.read_csv(GEN_DIR / "base_metadata.csv")
base_div = pd.read_csv(GEN_DIR / "base_lpips.csv")
print("train", len(train_df), "| test", len(test_df), "| prompts", len(prompts), "| sampler", sampler)

## 2.2 Training Run 2

One training step:
1. Encode the image to a 64×64×4 latent with the frozen VAE.
2. Encode the caption with the frozen CLIP text encoder.
3. Add random noise at a random timestep *t*.
4. The UNet (with LoRA) predicts that noise.
5. Loss = MSE between the predicted and the real noise. Only the LoRA weights are updated.

Memory savers: fp16 base weights, fp32 LoRA weights, gradient checkpointing, mixed precision.

`RETRAIN = False` reloads the adapter already saved in Drive and prints its saved training log. Set it to `True` to train again (about 75 min on a T4).

In [ ]:
TRAINED = (RUN_DIR / "stats.json").exists() and (RUN_DIR / "pytorch_lora_weights.safetensors").exists()
DO_TRAIN = RETRAIN or not TRAINED
print("adapter saved in Drive:", TRAINED, "| train now:", DO_TRAIN)

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
if DO_TRAIN:
    text_encoder = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to(device)
    vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to(device)
    noise_sched = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
    text_encoder.requires_grad_(False)
    vae.requires_grad_(False)
    print("frozen: text encoder + VAE")
else:
    print("not training: text encoder and VAE not loaded")

In [ ]:
class NarutoTrain(Dataset):
    def __init__(self, df):
        self.df = df
        self.tf = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        ids = tokenizer(r.text, padding="max_length", max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt").input_ids[0]
        return {"pixel_values": self.tf(Image.open(r.file).convert("RGB")), "input_ids": ids}

In [ ]:
train_ds = NarutoTrain(train_df)
len(train_ds)

In [ ]:
def make_lora_unet(rank):
    torch.manual_seed(SEED)
    unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16).to(device)
    unet.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=rank, init_lora_weights="gaussian",
                                target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
    cast_training_params(unet, dtype=torch.float32)
    unet.enable_gradient_checkpointing()
    return unet

In [ ]:
def train(unet, steps=STEPS, lr=LR, bs=BATCH):
    params = [p for p in unet.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)
    lr_sched = get_scheduler("cosine", opt, num_warmup_steps=100, num_training_steps=steps)
    scaler = torch.amp.GradScaler("cuda")
    loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=2,
                        generator=torch.Generator().manual_seed(SEED))

    log(f"{RUN} start | steps {steps} | batch {bs} | lr {lr} | trainable {sum(p.numel() for p in params):,}", file=TRAIN_LOG)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    losses = []
    step = 0
    unet.train()

    while step < steps:
        for batch in loader:
            with torch.no_grad():
                lat = vae.encode(batch["pixel_values"].to(device, torch.float16)).latent_dist.sample()
                lat = lat * vae.config.scaling_factor
                enc = text_encoder(batch["input_ids"].to(device))[0]

            noise = torch.randn_like(lat)
            t = torch.randint(0, noise_sched.config.num_train_timesteps, (lat.shape[0],), device=device).long()
            noisy = noise_sched.add_noise(lat, noise, t)

            with torch.autocast("cuda", dtype=torch.float16):
                pred = unet(noisy, t, enc).sample
            loss = F.mse_loss(pred.float(), noise.float())

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            lr_sched.step()
            opt.zero_grad(set_to_none=True)

            losses.append(loss.item())
            step += 1
            if step % 250 == 0:
                mem = torch.cuda.max_memory_allocated() / 1024 ** 3
                log(f"{RUN} step {step}/{steps} | loss {np.mean(losses[-250:]):.4f} | lr {lr_sched.get_last_lr()[0]:.2e} | "
                    f"{(time.time() - t0) / 60:.1f} min | peak {mem:.2f} GB", file=TRAIN_LOG)
            if step >= steps:
                break

    torch.cuda.synchronize()
    log(f"{RUN} done | {(time.time() - t0) / 60:.1f} min | peak {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB", file=TRAIN_LOG)
    return {
        "rank": RANK,
        "trainable_params": sum(p.numel() for p in params),
        "lr": lr, "steps": steps, "batch_size": bs,
        "peak_mem_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
        "train_sec": time.time() - t0,
        "gpu": torch.cuda.get_device_name(0),
        "losses": losses,
    }

In [ ]:
def save_run(unet, stats):
    lora_sd = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionPipeline.save_lora_weights(RUN_DIR, unet_lora_layers=lora_sd, safe_serialization=True)
    (RUN_DIR / "stats.json").write_text(json.dumps(stats))

In [ ]:
if not DO_TRAIN:
    stats = json.loads((RUN_DIR / "stats.json").read_text())
    print(f"loaded saved {RUN} from {RUN_DIR}\n")
    print(TRAIN_LOG.read_text() if TRAIN_LOG.exists() else "no saved training log")
else:
    unet = make_lora_unet(RANK)
    stats = train(unet)
    save_run(unet, stats)
    del unet
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
sorted(p.name for p in RUN_DIR.iterdir())

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(stats["losses"], alpha=0.25, label="loss per step")
plt.plot(pd.Series(stats["losses"]).rolling(100).mean(), label="rolling mean (100)")
plt.xlabel("step")
plt.ylabel("MSE loss")
plt.title(f"{RUN}: LoRA r={RANK} training loss")
plt.legend()
plt.show()

### Training table (Run 2)

In [ ]:
train_row = pd.Series({
    "method": f"LoRA r={stats['rank']}",
    "trainable_params": f"{stats['trainable_params']:,}",
    "learning_rate": stats["lr"],
    "steps": stats["steps"],
    "batch_size": stats["batch_size"],
    "peak_gpu_mem_GB": round(stats["peak_mem_gb"], 2),
    "train_time_min": round(stats["train_sec"] / 60, 1),
    "gpu_hours": round(stats["train_sec"] / 3600, 3),
    "gpu": stats["gpu"],
}, name=RUN)
train_row.to_frame().to_csv(EVAL / f"p2_{RUN}_training.csv")
log(f"{RUN} | " + " | ".join(f"{k} {v}" for k, v in train_row.items()))
train_row.to_frame()

In [ ]:
repo = f"{HF_USER}/naruto-sd15-lora-r{RANK}"
if UPLOAD:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    api = HfApi()
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=RUN_DIR, allow_patterns=["*.safetensors", "stats.json"])
    log("uploaded", RUN, "->", f"https://huggingface.co/{repo}")
else:
    print("UPLOAD is False: adapter saved only in", RUN_DIR)

## 2.3 Generation with Run 2

Same prompts, seeds (0 and 1) and sampler settings as the baseline, so the only difference from the base images is the LoRA adapter.

In [ ]:
if DO_TRAIN:
    del text_encoder, vae
gc.collect()
torch.cuda.empty_cache()

In [ ]:
GEN = {k: v for k, v in sampler.items() if k != "scheduler"}
META_FILE = GEN_DIR / f"{RUN}_metadata.csv"
GEN_STATS = GEN_DIR / f"{RUN}_gen_stats.json"
GEN_CACHED = False
if REUSE and not DO_TRAIN and META_FILE.exists() and GEN_STATS.exists():
    gen = pd.read_csv(META_FILE)
    GEN_CACHED = len(gen) == 40 and all(Path(f).exists() for f in gen.file)
print("reuse saved", RUN, "images from Drive:", GEN_CACHED, "| sampler", GEN)

In [ ]:
if not GEN_CACHED:
    pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None,
                                                   requires_safety_checker=False).to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.set_progress_bar_config(disable=True)
    pipe.load_lora_weights(str(RUN_DIR))

In [ ]:
if GEN_CACHED:
    gen_peak = json.loads(GEN_STATS.read_text())["peak_gb"]
    print("loaded", len(gen), RUN, "images from", GEN_DIR)
else:
    torch.cuda.reset_peak_memory_stats()
    rows = []
    for p in prompts:
        for seed in (0, 1):
            path = GEN_DIR / f"{RUN}_p{p['id']:02d}_s{seed}.png"
            t0 = time.time()
            img = pipe(p["prompt"], generator=torch.Generator(device).manual_seed(seed), **GEN).images[0]
            sec = time.time() - t0
            img.save(path)
            rows.append({"model": RUN, "prompt_id": p["id"], "category": p["category"],
                         "prompt": p["prompt"], "seed": seed, "file": str(path), "sec": sec})
    gen = pd.DataFrame(rows)
    gen_peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    gen.to_csv(META_FILE, index=False)
    GEN_STATS.write_text(json.dumps({"peak_gb": gen_peak, "version": 2}))
    log(f"{RUN} generation |", len(gen), "images | median sec/img", round(gen.sec.median(), 2))
print("images:", len(gen), "| median sec/img %.2f | peak GPU mem %.2f GB" % (gen.sec.median(), gen_peak))

In [ ]:
def show_pairs(ids, seed=0, title=None):
    fig, axes = plt.subplots(len(ids), 2, figsize=(7, 3.4 * len(ids)))
    for r, pid in enumerate(ids):
        for c, df in enumerate([base, gen]):
            f = df[(df.prompt_id == pid) & (df.seed == seed)].file.iloc[0]
            axes[r, c].imshow(Image.open(f))
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(["base", RUN][c], fontsize=12)
        axes[r, 0].set_ylabel(textwrap.fill(f"[{pid}] {prompts[pid]['prompt']}", 26), fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
grid_ids = [0, 1, 2, 3, 10, 12, 15, 16]
show_pairs(grid_ids, title=f"base vs {RUN} (seed 0)")

In [ ]:
show_pairs(grid_ids[:4], seed=1, title=f"base vs {RUN} (seed 1)")

## 2.4 Metrics for Run 2

In [ ]:
def load_uint8(files, size=None):
    ims = [Image.open(f).convert("RGB") for f in files]
    if size:
        ims = [im.resize((size, size), Image.BICUBIC) for im in ims]
    return torch.from_numpy(np.stack([np.asarray(im) for im in ims])).permute(0, 3, 1, 2)

In [ ]:
real = load_uint8(test_df.file, 299)
fake = load_uint8(gen[gen.category == "in_domain"].file, 299)
real.shape, fake.shape

In [ ]:
kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)), subsets=100).to(device)
torch.manual_seed(SEED)
for k in range(0, len(real), 32):
    kid.update(real[k:k + 32].to(device), real=True)
kid.update(fake.to(device), real=False)
kid_mu, kid_sd = [v.item() for v in kid.compute()]
log(f"{RUN} KID = {kid_mu:.4f} ± {kid_sd:.4f} (base was {baseline['KID in-domain (lower better)']:.4f})")

In [ ]:
clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to(device)

scores = []
for r in gen.itertuples():
    scores.append(clip_metric(load_uint8([r.file])[0].to(device), r.prompt).item())
    clip_metric.reset()
gen["clip"] = scores

In [ ]:
clip_cmp = pd.DataFrame({"base": base.groupby("category").clip.mean(), RUN: gen.groupby("category").clip.mean()})
clip_cmp[f"{RUN} - base"] = clip_cmp[RUN] - clip_cmp["base"]
clip_cmp.round(2)

In [ ]:
lp = lpips.LPIPS(net="alex").to(device)

def to_lp(f):
    x = np.asarray(Image.open(f).convert("RGB"), dtype=np.float32) / 127.5 - 1
    return torch.from_numpy(x).permute(2, 0, 1)[None].to(device)

In [ ]:
div = []
for pid, g in gen.groupby("prompt_id"):
    a, b = g.sort_values("seed").file.tolist()
    with torch.no_grad():
        d = lp(to_lp(a), to_lp(b)).item()
    div.append({"model": RUN, "prompt_id": pid, "category": g.category.iloc[0], "lpips": d})
div = pd.DataFrame(div)
pd.DataFrame({"base": base_div.groupby("category").lpips.mean(), RUN: div.groupby("category").lpips.mean()}).round(3)

In [ ]:
safety = None
safety_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

def nsfw_flags(paths, bs=32):
    global safety
    if safety is None:
        safety = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker").to(device)
    flags = []
    for k in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[k:k + bs]]
        clip_in = safety_proc(ims, return_tensors="pt").pixel_values.to(device)
        arr = np.stack([np.asarray(im, dtype=np.float32) / 255 for im in ims])
        _, has = safety(images=arr, clip_input=clip_in)
        flags += [bool(h) for h in has]
    return flags

if "nsfw" not in gen.columns:
    gen["nsfw"] = nsfw_flags(gen.file.tolist())
print(f"{RUN} generations flagged NSFW:", int(gen.nsfw.sum()), "/", len(gen))
gen[gen.nsfw][["prompt_id", "seed", "category", "prompt"]]

## Results: base vs Run 2

In [ ]:
run_res = pd.Series({
    "KID in-domain (lower better)": kid_mu,
    "KID std": kid_sd,
    "CLIP all": gen["clip"].mean(),
    "CLIP in_domain": gen[gen.category == "in_domain"]["clip"].mean(),
    "CLIP ood": gen[gen.category == "ood"]["clip"].mean(),
    "CLIP compositional": gen[gen.category == "compositional"]["clip"].mean(),
    "LPIPS diversity": div.lpips.mean(),
    "NSFW flagged": int(gen.nsfw.sum()),
    "median sec/img (gen)": gen.sec.median(),
    "GPU-hours (train)": stats["train_sec"] / 3600,
    "peak GPU mem GB (train)": stats["peak_mem_gb"],
    "GPU-hours (generate 40 imgs)": gen.sec.sum() / 3600,
    "peak GPU mem GB (generate)": gen_peak,
}, name=RUN)
results = pd.concat([baseline, run_res], axis=1).round(4)
results

In [ ]:
run_res.round(4).to_frame().to_csv(EVAL / f"p2_{RUN}_results.csv")
gen.to_csv(GEN_DIR / f"{RUN}_metadata.csv", index=False)
div.to_csv(GEN_DIR / f"{RUN}_lpips.csv", index=False)
log(f"{RUN} results\n" + results.to_string())

## Run 2 failure examples

In [ ]:
worst = gen.nsmallest(4, "clip")
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, row in zip(axes, worst.itertuples()):
    ax.imshow(Image.open(row.file))
    ax.axis("off")
    ax.set_title(textwrap.fill(f"CLIP {row.clip:.1f} | {row.prompt}", 34), fontsize=8)
fig.suptitle(f"{RUN}: lowest prompt match")
plt.tight_layout()
plt.show()

In [ ]:
show_pairs([p["id"] for p in prompts if p["category"] == "compositional"][:3], title=f"{RUN}: compositional prompts")

In [ ]:
low = div.nsmallest(2, "lpips")
fig, axes = plt.subplots(len(low), 2, figsize=(7, 3.4 * len(low)))
for r, row in enumerate(low.itertuples()):
    for s in (0, 1):
        f = gen[(gen.prompt_id == row.prompt_id) & (gen.seed == s)].file.iloc[0]
        axes[r, s].imshow(Image.open(f))
        axes[r, s].axis("off")
        axes[r, s].set_title(f"p{row.prompt_id} seed {s}  lpips={row.lpips:.3f}", fontsize=8)
fig.suptitle(f"{RUN}: least diverse prompts")
plt.tight_layout()
plt.show()

**Run 2 observations** (write in your own words after looking at the images):
- ...
- ...

In [ ]:
for f in [RUN_DIR / "pytorch_lora_weights.safetensors", RUN_DIR / "stats.json", EVAL / f"p2_{RUN}_training.csv",
          EVAL / f"p2_{RUN}_results.csv", GEN_DIR / f"{RUN}_metadata.csv", GEN_DIR / f"{RUN}_lpips.csv"]:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")

### Free GPU memory before the next section

In [ ]:
pipe = safety = kid = clip_metric = lp = None
gc.collect()
torch.cuda.empty_cache()
print("GPU memory still in use: %.2f GB" % (torch.cuda.memory_allocated() / 1024 ** 3))

# Section 4: Final comparison, human evaluation, safety

Reads the results saved by sections 1–3, picks the final run, and prints **one results table, baseline vs final**, with the four measurements.

## Load results from sections 1-3

In [ ]:
models = ["base", "run1", "run2"]
needed = [EVAL / "p2_prompts.json", EVAL / "p2_data_cleaning.json", EVAL / "p2_sampler.json",
          DATA / "train_clean.csv", DATA / "test.csv", EVAL / "p2_baseline_results.csv",
          EVAL / "p2_run1_results.csv", EVAL / "p2_run2_results.csv",
          EVAL / "p2_run1_training.csv", EVAL / "p2_run2_training.csv"]
needed += [GEN_DIR / f"{m}_metadata.csv" for m in models] + [GEN_DIR / f"{m}_lpips.csv" for m in models]
for f in needed:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")
assert all(f.exists() for f in needed), "run sections 1-3 first"

In [ ]:
prompts = json.loads((EVAL / "p2_prompts.json").read_text())
cleaning = json.loads((EVAL / "p2_data_cleaning.json").read_text())
sampler = json.loads((EVAL / "p2_sampler.json").read_text())
train_df = pd.read_csv(DATA / "train_clean.csv")
test_df = pd.read_csv(DATA / "test.csv")
gen = pd.concat([pd.read_csv(GEN_DIR / f"{m}_metadata.csv") for m in models], ignore_index=True)
div = pd.concat([pd.read_csv(GEN_DIR / f"{m}_lpips.csv") for m in models], ignore_index=True)
gen.groupby("model").size()

In [ ]:
results = pd.concat([
    pd.read_csv(EVAL / "p2_baseline_results.csv", index_col=0)["base"],
    pd.read_csv(EVAL / "p2_run1_results.csv", index_col=0)["run1"],
    pd.read_csv(EVAL / "p2_run2_results.csv", index_col=0)["run2"],
], axis=1)
results

## 2.2 Training table (both runs)

In [ ]:
train_table = pd.concat([
    pd.read_csv(EVAL / "p2_run1_training.csv", index_col=0)["run1"],
    pd.read_csv(EVAL / "p2_run2_training.csv", index_col=0)["run2"],
], axis=1).T
train_table.to_csv(EVAL / "p2_training_table.csv")
train_table

## Choosing the final run

Default: the run with the lowest in-domain KID (best domain style). To choose the other run for reasons you explain below (prompt match, diversity, size), set `FINAL_OVERRIDE = "run1"` or `"run2"` in the config cell.

In [ ]:
kid_row = "KID in-domain (lower better)"
FINAL = FINAL_OVERRIDE or results.loc[kid_row, ["run1", "run2"]].astype(float).idxmin()
log("final run:", FINAL, "| KID run1 %.4f | run2 %.4f" % (results.loc[kid_row, "run1"], results.loc[kid_row, "run2"]))

## 2.3 Side-by-side grids: base vs Run 1 vs Run 2

In [ ]:
def pick(m, pid, seed):
    return gen[(gen.model == m) & (gen.prompt_id == pid) & (gen.seed == seed)].file.iloc[0]

def show_grid(ids, seed=0, title=None):
    fig, axes = plt.subplots(len(ids), 3, figsize=(10, 3.3 * len(ids)))
    axes = np.atleast_2d(axes)
    for r, pid in enumerate(ids):
        for c, m in enumerate(models):
            axes[r, c].imshow(Image.open(pick(m, pid, seed)))
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(m, fontsize=13)
        axes[r, 0].set_ylabel(textwrap.fill(f"[{pid}] {prompts[pid]['prompt']}", 28), fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
grid_ids = [0, 1, 2, 3, 10, 12, 15, 16]
show_grid(grid_ids, title="base vs run1 vs run2 (seed 0)")

In [ ]:
show_grid(grid_ids[:4], seed=1, title="base vs run1 vs run2 (seed 1)")

## Human evaluation (blinded, final run vs base)

25 pairs, with left/right randomized by seed 266. Raters only see `pair_XX.png` and the prompt; the answer key stays hidden in `eval/p2_human_key.csv`. Each rater picks L or R for **style** (more like Naruto) and for **prompt_match**. At least 3 raters: both team members plus one outsider. Fill `eval/p2_human_ratings.csv`, then rerun from the ratings cell.

In [ ]:
KEY_FILE = EVAL / "p2_human_key.csv"
KEY_CACHED = False
if REUSE and KEY_FILE.exists():
    key = pd.read_csv(KEY_FILE)
    KEY_CACHED = "final_model" in key.columns and key.final_model.iloc[0] == FINAL and len(key) == 25 \
        and all((HUMAN_DIR / f"pair_{k:02d}.png").exists() for k in key.pair)
print("reuse saved human-eval pairs from Drive:", KEY_CACHED)

In [ ]:
rng = np.random.default_rng(SEED)
pairs = gen[gen.model == "base"][["prompt_id", "seed", "prompt"]].sample(25, random_state=SEED).reset_index(drop=True)

new_key = []
for k, r in pairs.iterrows():
    if KEY_CACHED:
        break
    final_left = bool(rng.integers(2))
    a = pick(FINAL, r.prompt_id, r.seed)
    b = pick("base", r.prompt_id, r.seed)
    left, right = (a, b) if final_left else (b, a)

    canvas = Image.new("RGB", (2 * RES + 20, RES), "white")
    canvas.paste(Image.open(left), (0, 0))
    canvas.paste(Image.open(right), (RES + 20, 0))
    canvas.save(HUMAN_DIR / f"pair_{k:02d}.png")

    new_key.append({"pair": k, "prompt_id": r.prompt_id, "seed": r.seed, "prompt": r.prompt,
                    "final_side": "L" if final_left else "R", "final_model": FINAL})

In [ ]:
if not KEY_CACHED:
    key = pd.DataFrame(new_key)
    key.to_csv(KEY_FILE, index=False)
    key[["pair", "prompt"]].to_csv(HUMAN_DIR / "rater_sheet.csv", index=False)
    print("25 new pairs written to", HUMAN_DIR)
else:
    print("using the 25 saved pairs in", HUMAN_DIR)
print("final run shown on left in", (key.final_side == "L").sum(), "of 25 pairs")

In [ ]:
ratings_file = EVAL / "p2_human_ratings.csv"
if not ratings_file.exists():
    pd.DataFrame(columns=["rater", "pair", "style", "prompt_match"]).to_csv(ratings_file, index=False)
ratings = pd.read_csv(ratings_file).dropna()
print("ratings:", len(ratings), "rows from", ratings.rater.nunique(), "raters:", sorted(ratings.rater.unique()))

In [ ]:
def fleiss_kappa(df, col):
    t = df.pivot_table(index="pair", columns=col, values="rater", aggfunc="count", fill_value=0)
    n = t.sum(1).iloc[0]
    p_j = t.sum(0) / t.values.sum()
    P_i = ((t ** 2).sum(1) - n) / (n * (n - 1))
    Pe = (p_j ** 2).sum()
    return (P_i.mean() - Pe) / (1 - Pe) if Pe < 1 else 1.0

In [ ]:
human = {"style": np.nan, "prompt_match": np.nan}
kappa = {"style": np.nan, "prompt_match": np.nan}
if len(ratings):
    ratings = ratings.merge(key[["pair", "final_side"]], on="pair")
    for c in ["style", "prompt_match"]:
        ratings[f"{c}_final_wins"] = ratings[c].str.strip().str.upper() == ratings.final_side
        human[c] = ratings[f"{c}_final_wins"].mean()
        kappa[c] = fleiss_kappa(ratings, f"{c}_final_wins")
log("human eval |", len(ratings), "ratings,", ratings.rater.nunique() if len(ratings) else 0, "raters | win rate of",
    FINAL, "vs base", {k: round(float(v), 3) for k, v in human.items()}, "| Fleiss kappa", {k: round(float(v), 3) for k, v in kappa.items()})

In [ ]:
if len(ratings):
    print(ratings.groupby("rater")[["style_final_wins", "prompt_match_final_wins"]].mean().round(2))

### AI judge (automatic, not a human rater)

CLIP ViT-L/14 judges the same 25 blinded pairs:
- **style:** the image whose CLIP embedding is closer to the average embedding of the real Naruto test images
- **prompt match:** the image with the higher CLIP similarity to the prompt

This is reported separately from the human evaluation and does not replace it. When human ratings exist, we also report how often the AI judge agrees with the human majority.

In [ ]:
judge_m = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
judge_p = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

@torch.no_grad()
def judge_img(files, bs=32):
    out = []
    for k in range(0, len(files), bs):
        ims = [Image.open(f).convert("RGB") for f in files[k:k + bs]]
        x = judge_p(images=ims, return_tensors="pt").pixel_values.to(device)
        out.append(F.normalize(judge_m.get_image_features(pixel_values=x).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def judge_txt(texts):
    t = judge_p(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    return F.normalize(judge_m.get_text_features(**t).float(), dim=-1)

In [ ]:
style_ref = F.normalize(judge_img(test_df.file.tolist()).mean(0, keepdim=True), dim=-1)[0]

ai = []
for r in key.itertuples():
    e = judge_img([pick(FINAL, r.prompt_id, r.seed), pick("base", r.prompt_id, r.seed)])
    t = judge_txt([r.prompt])[0]
    other_side = "R" if r.final_side == "L" else "L"
    style_final = bool(e[0] @ style_ref > e[1] @ style_ref)
    match_final = bool(e[0] @ t > e[1] @ t)
    ai.append({"pair": r.pair, "ai_style": r.final_side if style_final else other_side,
               "ai_prompt_match": r.final_side if match_final else other_side,
               "style_final_wins": style_final, "prompt_match_final_wins": match_final,
               "style_sim_final": (e[0] @ style_ref).item(), "style_sim_base": (e[1] @ style_ref).item(),
               "text_sim_final": (e[0] @ t).item(), "text_sim_base": (e[1] @ t).item()})
ai = pd.DataFrame(ai)
ai.to_csv(EVAL / "p2_ai_judge.csv", index=False)
ai_win = {"style": ai.style_final_wins.mean(), "prompt_match": ai.prompt_match_final_wins.mean()}
log("AI judge (CLIP ViT-L/14) | win rate of", FINAL, "vs base", {k: round(float(v), 3) for k, v in ai_win.items()})
ai.head()

In [ ]:
ai_agree = {"style": np.nan, "prompt_match": np.nan}
if len(ratings):
    for c in ["style", "prompt_match"]:
        maj = ratings.assign(v=ratings[c].str.strip().str.upper()).groupby("pair").v.agg(lambda x: x.value_counts().index[0])
        both = ai.set_index("pair")[f"ai_{c}"].to_frame().join(maj.rename("human"), how="inner")
        ai_agree[c] = (both[f"ai_{c}"] == both.human).mean()
    log("AI judge agreement with human majority", {k: round(float(v), 3) for k, v in ai_agree.items()})
else:
    print("no human ratings yet: agreement with humans will be computed after they are added")

In [ ]:
judge_m = None
gc.collect()
torch.cuda.empty_cache()

## Results table: baseline vs final (four measurements)

Every number below was computed earlier in this notebook, saved to Drive, and is printed again here. The base model's human win rate is the 50% reference: it is what the final run is compared against.

In [ ]:
f = results[FINAL]
b = results["base"]
table = pd.DataFrame([
    ("Primary", "KID in-domain vs test images (lower better)", b[kid_row], f[kid_row]),
    ("Primary", "KID std (100 subsets)", b["KID std"], f["KID std"]),
    ("Primary", "CLIP score, all 20 prompts (higher better)", b["CLIP all"], f["CLIP all"]),
    ("Primary", "CLIP score, in-domain", b["CLIP in_domain"], f["CLIP in_domain"]),
    ("Primary", "CLIP score, out-of-domain", b["CLIP ood"], f["CLIP ood"]),
    ("Primary", "CLIP score, compositional", b["CLIP compositional"], f["CLIP compositional"]),
    ("Secondary", "LPIPS diversity between seeds (higher = more varied)", b["LPIPS diversity"], f["LPIPS diversity"]),
    ("Human", "Win rate vs base, domain style (25 pairs)", 0.5, human["style"]),
    ("Human", "Win rate vs base, prompt match (25 pairs)", 0.5, human["prompt_match"]),
    ("AI judge (not human)", "CLIP judge win rate vs base, domain style", 0.5, ai_win["style"]),
    ("AI judge (not human)", "CLIP judge win rate vs base, prompt match", 0.5, ai_win["prompt_match"]),
    ("AI judge (not human)", "Agreement with human majority, style", np.nan, ai_agree["style"]),
    ("AI judge (not human)", "Agreement with human majority, prompt match", np.nan, ai_agree["prompt_match"]),
    ("Efficiency", "GPU-hours, training", b["GPU-hours (train)"], f["GPU-hours (train)"]),
    ("Efficiency", "Peak GPU memory GB, training", b["peak GPU mem GB (train)"], f["peak GPU mem GB (train)"]),
    ("Efficiency", "GPU-hours, generating 40 images", b["GPU-hours (generate 40 imgs)"], f["GPU-hours (generate 40 imgs)"]),
    ("Efficiency", "Peak GPU memory GB, generation", b["peak GPU mem GB (generate)"], f["peak GPU mem GB (generate)"]),
], columns=["measurement", "metric", "baseline (base SD 1.5)", f"final ({FINAL})"])
table = table.set_index(["measurement", "metric"]).astype(float).round(4)
table.to_csv(EVAL / "p2_results.csv")
log("final results table\n" + table.to_string())
table

### All three models (for the report appendix)

In [ ]:
results.astype(float).round(4).to_csv(EVAL / "p2_results_all_models.csv")
results.astype(float).round(4)

## Failure modes

In [ ]:
ood_ids = [p["id"] for p in prompts if p["category"] == "ood"]
show_grid(ood_ids[:3], title="out-of-domain prompts")

In [ ]:
comp_ids = [p["id"] for p in prompts if p["category"] == "compositional"]
show_grid(comp_ids[:3], title="compositional prompts")

In [ ]:
worst = gen[gen.model != "base"].nsmallest(4, "clip")
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, row in zip(axes, worst.itertuples()):
    ax.imshow(Image.open(row.file))
    ax.axis("off")
    ax.set_title(textwrap.fill(f"{row.model} | CLIP {row.clip:.1f} | {row.prompt}", 34), fontsize=8)
fig.suptitle("lowest prompt match (fine-tuned runs)")
plt.tight_layout()
plt.show()

In [ ]:
low = div[div.model != "base"].nsmallest(3, "lpips")
fig, axes = plt.subplots(len(low), 2, figsize=(7, 3.4 * len(low)))
for r, row in enumerate(low.itertuples()):
    for s in (0, 1):
        axes[r, s].imshow(Image.open(pick(row.model, row.prompt_id, s)))
        axes[r, s].axis("off")
        axes[r, s].set_title(f"{row.model} p{row.prompt_id} seed {s}  lpips={row.lpips:.3f}", fontsize=8)
fig.suptitle("least diverse prompts (fine-tuned runs)")
plt.tight_layout()
plt.show()

In [ ]:
delta = gen.pivot_table(index="category", columns="model", values="clip", aggfunc="mean")
delta["run1 - base"] = delta["run1"] - delta["base"]
delta["run2 - base"] = delta["run2"] - delta["base"]
delta.round(2)

### Dark-border artifact check

Some training images have black borders. This measures how many generations have a dark band (mean brightness < 25 of 255) on the left or right 6% of the image.

In [ ]:
def dark_edge(f, frac=0.06, thr=25):
    x = np.asarray(Image.open(f).convert("L"), dtype=np.float32)
    w = int(x.shape[1] * frac)
    return bool(min(x[:, :w].mean(), x[:, -w:].mean()) < thr)

gen["dark_edge"] = [dark_edge(f) for f in gen.file]
train_dark = np.mean([dark_edge(f) for f in train_df.file])
print("training images with a dark side band: %.1f%%" % (100 * train_dark))
gen.groupby("model").dark_edge.sum()

In [ ]:
dark = gen[(gen.model != "base") & gen.dark_edge].head(4)
if len(dark):
    fig, axes = plt.subplots(1, len(dark), figsize=(4 * len(dark), 4.4))
    axes = np.atleast_1d(axes)
    for ax, row in zip(axes, dark.itertuples()):
        ax.imshow(Image.open(row.file))
        ax.axis("off")
        ax.set_title(textwrap.fill(f"{row.model} | {row.prompt}", 32), fontsize=8)
    fig.suptitle("dark side-band artifact")
    plt.tight_layout()
    plt.show()
else:
    print("no dark-border generations found")

### Failure modes and choice of final run (written from the numbers above)

In [ ]:
r = results.astype(float)
other = "run1" if FINAL == "run2" else "run2"
stats = {m: json.loads((RUNS_DIR / m / "stats.json").read_text()) for m in ["run1", "run2"]}
dark_n = gen.groupby("model").dark_edge.sum()
fin_div = div[div.model == FINAL].merge(div[div.model == "base"][["prompt_id", "lpips"]], on="prompt_id", suffixes=("", "_base"))
fin_gen = gen[gen.model == FINAL]
worst_comp = fin_gen[fin_gen.category == "compositional"].nsmallest(1, "clip").iloc[0]
worst_in = fin_gen[fin_gen.category == "in_domain"].nsmallest(1, "clip").iloc[0]

failures = []
d_in = r.loc["CLIP in_domain", FINAL] - r.loc["CLIP in_domain", "base"]
if d_in < 0:
    failures.append(
        f"Weaker prompt alignment on in-domain prompts. CLIP in-domain falls from {r.loc['CLIP in_domain', 'base']:.2f} (base) "
        f"to {r.loc['CLIP in_domain', FINAL]:.2f} ({d_in:+.2f}), so images follow the caption text less closely after fine-tuning; "
        f"the worst case is \"{worst_in.prompt}\" (CLIP {worst_in['clip']:.1f}).")
d_ood = r.loc["CLIP ood", FINAL] - r.loc["CLIP ood", "base"]
if d_ood < 0:
    failures.append(
        f"Style leaking into out-of-domain prompts. CLIP on out-of-domain prompts drops by {abs(d_ood):.2f} "
        f"({r.loc['CLIP ood', 'base']:.2f} to {r.loc['CLIP ood', FINAL]:.2f}), so prompts with no Naruto content "
        f"(cars, lighthouse, fruit) follow the text slightly less well than with the base model.")
if worst_comp["clip"] < fin_gen["clip"].median():
    failures.append(
        f"Weak compositional prompts. The lowest-scoring compositional prompt for {FINAL} is \"{worst_comp.prompt}\" "
        f"(CLIP {worst_comp['clip']:.1f}, below the {FINAL} median of {fin_gen['clip'].median():.1f}); compositional mean "
        f"{r.loc['CLIP compositional', FINAL]:.2f} vs base {r.loc['CLIP compositional', 'base']:.2f}. See the compositional grid above.")
d_div = r.loc["LPIPS diversity", FINAL] - r.loc["LPIPS diversity", "base"]
if d_div < 0:
    lowest = fin_div.nsmallest(1, "lpips").iloc[0]
    failures.append(
        f"Less diversity between seeds. Mean LPIPS between seed 0 and seed 1 falls from {r.loc['LPIPS diversity', 'base']:.3f} "
        f"to {r.loc['LPIPS diversity', FINAL]:.3f}; the least varied prompt is p{int(lowest.prompt_id)} "
        f"(LPIPS {lowest.lpips:.3f} vs {lowest.lpips_base:.3f} for base).")
if dark_n.get(FINAL, 0) > dark_n.get("base", 0):
    failures.append(
        f"Dark side-band artifact. {int(dark_n[FINAL])}/40 {FINAL} images have a black band on one side, "
        f"against {int(dark_n['base'])}/40 for base" + (f"; {100 * train_dark:.1f}% of training images have such a band, so the model learned it from the data."
                                                         if train_dark > 0 else "."))

kid_f, kid_o = r.loc[kid_row, FINAL], r.loc[kid_row, other]
choice = [
    f"Final run: {FINAL} (LoRA rank {stats[FINAL]['rank']}, {stats[FINAL]['trainable_params']:,} trainable parameters).",
    f"Domain style: KID {kid_f:.4f} for {FINAL} vs {kid_o:.4f} for {other} and {r.loc[kid_row, 'base']:.4f} for base "
    f"({r.loc[kid_row, 'base'] / kid_f:.1f}x lower than base).",
    f"Prompt match (CLIP, all 20 prompts): {r.loc['CLIP all', FINAL]:.2f} for {FINAL} vs {r.loc['CLIP all', other]:.2f} for {other}.",
    f"Diversity (LPIPS): {r.loc['LPIPS diversity', FINAL]:.3f} for {FINAL} vs {r.loc['LPIPS diversity', other]:.3f} for {other}.",
    f"Cost: {stats[FINAL]['train_sec'] / 3600:.2f} vs {stats[other]['train_sec'] / 3600:.2f} GPU-hours, "
    f"{stats[FINAL]['peak_mem_gb']:.2f} vs {stats[other]['peak_mem_gb']:.2f} GB peak memory.",
]
cost_ratio = stats[FINAL]["train_sec"] / stats[other]["train_sec"]
if kid_f <= kid_o and r.loc["CLIP all", FINAL] < r.loc["CLIP all", other]:
    choice.append(f"Trade-off: {FINAL} has the lowest KID (best Naruto style, the main goal of the fine-tune) but "
                  f"{r.loc['CLIP all', other] - r.loc['CLIP all', FINAL]:.2f} lower CLIP than {other}; it took "
                  f"{cost_ratio:.2f}x the training time of {other}.")
elif kid_f > kid_o:
    choice.append(f"Trade-off: {other} has a lower KID, but {FINAL} was chosen for better prompt match and diversity "
                  f"with fewer trainable parameters.")
else:
    choice.append(f"{FINAL} is at least as good as {other} on both style and prompt match.")
if not np.isnan(human["style"]):
    choice.append(f"Human evaluation: raters preferred {FINAL} over base in {100 * human['style']:.0f}% of votes for style "
                  f"and {100 * human['prompt_match']:.0f}% for prompt match.")
else:
    choice.append("Human evaluation: not filled in yet (add eval/p2_human_ratings.csv and rerun).")
choice.append(f"AI judge (CLIP, automatic, not a human rater): {FINAL} preferred over base in {100 * ai_win['style']:.0f}% "
              f"of pairs for style and {100 * ai_win['prompt_match']:.0f}% for prompt match"
              + (f"; it agrees with the human majority on {100 * ai_agree['style']:.0f}% (style) and "
                 f"{100 * ai_agree['prompt_match']:.0f}% (prompt) of pairs." if not np.isnan(ai_agree["style"]) else "."))

analysis = "## Failure modes\n\n" + "\n".join(f"{i}. {t}" for i, t in enumerate(failures, 1))
analysis += "\n\n## Chosen run and why\n\n" + "\n".join(f"- {t}" for t in choice)
print(analysis)

## Safety (C4)

1. **License.** Printed below. The images show copyrighted Naruto characters, so the model and its outputs are for non-commercial research only. SD 1.5 is under CreativeML OpenRAIL-M, which forbids harmful uses.
2. **NSFW filter.** The official SD safety checker (`CompVis/stable-diffusion-safety-checker`, CLIP-based) was run on all training images (flagged ones removed) and on all 120 generations (flag stored in the dataset metadata).
3. **Memorization** (Carlini et al., 2023). For the 10 in-domain generations of the final run (seed 0), we find the nearest training image by CLIP ViT-L/14 cosine similarity. Real *test* images give the "normal" similarity for unseen images; a generation far above that level would be a likely copy.

In [ ]:
print("dataset license :", cleaning["license"])
print("base model      : creativeml-openrail-m")
print("NSFW removed from training:", cleaning["train_nsfw_removed"], "of", cleaning["train"] + cleaning["train_nsfw_removed"])
print("NSFW flagged in generations:", gen.groupby("model").nsfw.sum().to_dict())

In [ ]:
clip_m = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
clip_p = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

@torch.no_grad()
def embed(files, bs=64):
    out = []
    for k in range(0, len(files), bs):
        ims = [Image.open(f).convert("RGB") for f in files[k:k + bs]]
        x = clip_p(images=ims, return_tensors="pt").pixel_values.to(device)
        out.append(F.normalize(clip_m.get_image_features(pixel_values=x).float(), dim=-1))
    return torch.cat(out)

In [ ]:
EMB_FILE = WORK / "p2_clip_embeddings.pt"
if REUSE and EMB_FILE.exists():
    saved = torch.load(EMB_FILE)
    train_emb, test_emb = saved["train"].to(device), saved["test"].to(device)
    print("loaded CLIP embeddings from", EMB_FILE)
else:
    train_emb = embed(train_df.file.tolist())
    test_emb = embed(test_df.file.tolist())
    torch.save({"train": train_emb.cpu(), "test": test_emb.cpu()}, EMB_FILE)
    print("saved CLIP embeddings to", EMB_FILE)
test_nn = (test_emb @ train_emb.T).max(1).values.cpu().numpy()
print("test -> train nearest CLIP sim: median %.3f, max %.3f" % (np.median(test_nn), test_nn.max()))
print("test images with a near-duplicate in train (sim > 0.95):", int((test_nn > 0.95).sum()), "/", len(test_nn))

In [ ]:
mem = gen[(gen.model == FINAL) & (gen.category == "in_domain") & (gen.seed == 0)].reset_index(drop=True)
best, idx = (embed(mem.file.tolist()) @ train_emb.T).max(1)
mem["nn_file"] = train_df.file.values[idx.cpu().numpy()]
mem["nn_sim"] = best.cpu().numpy()
mem[["prompt_id", "prompt", "nn_sim"]].sort_values("nn_sim", ascending=False)

In [ ]:
top3 = mem.nlargest(3, "nn_sim")
fig, axes = plt.subplots(3, 2, figsize=(7, 10.5))
for r, row in enumerate(top3.itertuples()):
    axes[r, 0].imshow(Image.open(row.file))
    axes[r, 0].set_title(f"{FINAL} generation p{row.prompt_id}", fontsize=9)
    axes[r, 1].imshow(Image.open(row.nn_file))
    axes[r, 1].set_title(f"closest training image  sim={row.nn_sim:.3f}", fontsize=9)
    axes[r, 0].axis("off")
    axes[r, 1].axis("off")
fig.suptitle("3 closest generation / training pairs")
plt.tight_layout()
plt.show()

In [ ]:
n_high = int((mem.nn_sim > test_nn.max()).sum())
n_dup_level = int((mem.nn_sim > 0.95).sum())
if n_dup_level > 0:
    verdict = f"possible memorization: {n_dup_level} generation(s) have CLIP sim > 0.95 to a training image"
elif n_high > 0:
    verdict = f"borderline: {n_high} generation(s) are closer to training data than any real test image"
else:
    verdict = "no sign of memorization: generations are no closer to the training set than unseen test images are"
log("memorization | max gen->train %.3f | max test->train %.3f |" % (mem.nn_sim.max(), test_nn.max()), verdict)

In [ ]:
memo = (
    f"For the 10 in-domain generations of {FINAL} (seed 0), the closest training image has CLIP similarity "
    f"between {mem.nn_sim.min():.3f} and {mem.nn_sim.max():.3f} (median {mem.nn_sim.median():.3f}). "
    f"Real test images, which the model never saw, reach up to {test_nn.max():.3f} (median {np.median(test_nn):.3f}). "
    f"{int((mem.nn_sim > test_nn.max()).sum())} of 10 generations are closer to the training set than the closest test image, "
    f"and {int((mem.nn_sim > 0.95).sum())} exceed 0.95. Conclusion: {verdict}. "
    f"Note: {int((test_nn > 0.95).sum())} test images have a near-duplicate in the training set (sim > 0.95), "
    f"a small overlap the perceptual-hash de-duplication did not catch."
)
print(memo)

In [ ]:
nsfw_rate = cleaning["train_nsfw_removed"] / (cleaning["train"] + cleaning["train_nsfw_removed"])
safety_txt = (
    f"License: {cleaning['license']}. Base model: CreativeML OpenRAIL-M. "
    f"NSFW: the SD safety checker removed {cleaning['train_nsfw_removed']} of "
    f"{cleaning['train'] + cleaning['train_nsfw_removed']} training images ({100 * nsfw_rate:.1f}%). "
)
if nsfw_rate > 0.10:
    safety_txt += ("This rate is high for an anime dataset; the CLIP-based checker is known to be conservative on drawn "
                   "characters, so some flags are likely false positives (not verified by hand). ")
safety_txt += "Generations flagged: " + ", ".join(f"{m} {int(n)}/40" for m, n in gen.groupby("model").nsfw.sum().items()) + "."
print(safety_txt)

In [ ]:
report = analysis + "\n\n## Memorization\n\n" + memo + "\n\n## Safety\n\n" + safety_txt
(EVAL / "p2_analysis.md").write_text(report)
log("analysis written to", EVAL / "p2_analysis.md")

## Image dataset (120 images with prompt, seed and model)

In [ ]:
gen.drop(columns="sec").to_csv(GEN_DIR / "metadata.csv", index=False)
hf_gen = HFDataset.from_pandas(
    gen.assign(image=gen.file)[["image", "model", "prompt_id", "category", "prompt", "seed", "clip", "nsfw"]]
).cast_column("image", HFImage())
hf_gen.save_to_disk(str(GEN_DIR / "hf_dataset"))
hf_gen

In [ ]:
GEN_REPO = f"{HF_USER}/naruto-sd15-p2-generations"
if UPLOAD:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    hf_gen.push_to_hub(GEN_REPO, private=True)
    log("uploaded images ->", f"https://huggingface.co/datasets/{GEN_REPO}")
else:
    print("UPLOAD is False: dataset saved only in", GEN_DIR / "hf_dataset")

## Architecture / training diagram (for the IEEE report)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.6))
ax.set_xlim(0, 13)
ax.set_ylim(0, 4.6)
ax.axis("off")

def box(x, y, w, h, text, color):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="black", lw=1.2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9)

def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.3))

frozen = "#dfe6ee"
trained = "#f6c56b"
box(0.2, 3.0, 1.8, 1.0, "Image\n512×512", "white")
box(0.2, 0.6, 1.8, 1.0, "BLIP caption", "white")
box(2.6, 3.0, 1.9, 1.0, "VAE encoder\n(frozen)", frozen)
box(2.6, 0.6, 1.9, 1.0, "CLIP text encoder\n(frozen)", frozen)
box(5.1, 3.0, 1.9, 1.0, "+ noise at\ntimestep t", "white")
box(7.6, 1.7, 2.4, 1.6, "UNet (frozen)\n+ LoRA on q,k,v,out\n(trained, r=4 / 32)", trained)
box(10.6, 1.9, 2.2, 1.2, "MSE(pred noise,\nreal noise)", "white")
arrow(2.0, 3.5, 2.6, 3.5)
arrow(4.5, 3.5, 5.1, 3.5)
arrow(7.0, 3.5, 7.6, 2.9)
arrow(2.0, 1.1, 2.6, 1.1)
arrow(4.5, 1.1, 7.6, 2.1)
arrow(10.0, 2.5, 10.6, 2.5)
ax.annotate("", xy=(8.8, 1.7), xytext=(11.7, 1.9),
            arrowprops=dict(arrowstyle="->", lw=1.3, ls="--", connectionstyle="arc3,rad=-0.4"))
ax.text(10.6, 0.75, "backprop: LoRA weights only", fontsize=8, style="italic")
ax.text(4.8, 3.65, "latent\n64×64×4", fontsize=7, ha="center")
ax.set_title("Part 2 training pipeline: Stable Diffusion 1.5 + LoRA", fontsize=11)
plt.savefig(EVAL / "p2_architecture.png", dpi=200, bbox_inches="tight")
plt.show()

## Model card for the final run

In [ ]:
fc = json.loads((RUNS_DIR / FINAL / "stats.json").read_text())
repo = f"{HF_USER}/naruto-sd15-lora-r{fc['rank']}"
hs = "-" if np.isnan(human["style"]) else f"{human['style']:.2f}"
hp = "-" if np.isnan(human["prompt_match"]) else f"{human['prompt_match']:.2f}"

card = f"""---
license: creativeml-openrail-m
base_model: {BASE}
datasets:
- {DATASET}
tags: [stable-diffusion, lora, text-to-image, diffusers]
---

# Naruto-style LoRA for Stable Diffusion 1.5 (rank {fc['rank']})

LoRA adapter on the SD 1.5 UNet attention layers (q, k, v, out), trained on {len(train_df)} cleaned images of
`{DATASET}` (90/10 split, seed {SEED}). DATA 266 Lab 2, Part 2.

## Training
| | |
|---|---|
| trainable params | {fc['trainable_params']:,} |
| learning rate | {fc['lr']} (cosine, 100 warmup) |
| steps / batch | {fc['steps']} / {fc['batch_size']} |
| resolution | {RES} |
| GPU | {fc['gpu']} |
| GPU-hours | {fc['train_sec'] / 3600:.3f} |
| peak memory | {fc['peak_mem_gb']:.2f} GB |

## Evaluation ({sampler['scheduler']}, {sampler['num_inference_steps']} steps, CFG {sampler['guidance_scale']}, seeds 0 and 1)
| metric | base | this model |
|---|---|---|
| KID in-domain (lower better) | {b[kid_row]:.4f} | {f[kid_row]:.4f} |
| CLIP score (20 prompts) | {b['CLIP all']:.2f} | {f['CLIP all']:.2f} |
| LPIPS diversity | {b['LPIPS diversity']:.3f} | {f['LPIPS diversity']:.3f} |
| human win rate vs base (style / prompt) | - | {hs} / {hp} |

KID uses only 20 generated images, so it is noisy.

## Use
```python
from diffusers import StableDiffusionPipeline
import torch
pipe = StableDiffusionPipeline.from_pretrained("{BASE}", torch_dtype=torch.float16).to("cuda")
pipe.load_lora_weights("{repo}")
pipe("a man with blue hair and a headband").images[0]
```

## Limitations and safety
- Training captions are BLIP-generated and generic; prompt following on out-of-domain and multi-character prompts is weak.
- Dataset license: {cleaning['license']}.
- Training images were screened with the SD safety checker ({cleaning['train_nsfw_removed']} removed); a CLIP nearest-neighbour memorization check found: {verdict}.
"""
print(card)

In [ ]:
(RUNS_DIR / FINAL / "README.md").write_text(card)
(EVAL / "p2_model_card.md").write_text(card)
if UPLOAD:
    ModelCard(card).push_to_hub(repo)
    log("model card pushed to", repo)
else:
    print("UPLOAD is False: model card saved as", EVAL / "p2_model_card.md")

## Part 2 deliverables checklist

In [ ]:
checks = {
    "cleaning report": (EVAL / "p2_data_cleaning.json").exists(),
    "20 fixed prompts": len(prompts) == 20,
    "40 images per model": bool((gen.groupby("model").size() == 40).all()),
    "120 images total": len(gen) == 120,
    "training table": (EVAL / "p2_training_table.csv").exists(),
    "results table (baseline vs final)": (EVAL / "p2_results.csv").exists(),
    "architecture diagram": (EVAL / "p2_architecture.png").exists(),
    "human key": (EVAL / "p2_human_key.csv").exists(),
    "3+ raters": len(ratings) > 0 and ratings.rater.nunique() >= 3,
    "AI judge": (EVAL / "p2_ai_judge.csv").exists(),
    "model card": (EVAL / "p2_model_card.md").exists(),
    "analysis notes": (EVAL / "p2_analysis.md").exists(),
    "uploaded to Hugging Face": UPLOAD,
}
log("checklist", checks)
pd.Series(checks, name="done")